In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json
import os

print("✅ Analysis environment ready!")

In [ ]:
def analyze_confusion_matrix(df, method_name):
    """
    ANALYSIS 1: Confusion Matrix & Class Accuracy
    ---------------------------------------------
    What it does: Plots a matrix comparing ground truth labels to the model's predictions.
    
    How to interpret for Report: 
    - The diagonal (top-left to bottom-right) shows your correct predictions.
    - Dark squares OFF the diagonal indicate common failure modes. 
    - Example: If the 'tie' row has a high number in the 'A' column, your model has a 
      "bias against ambiguity"—it feels forced to pick a winner (A) even when humans 
      agreed the responses were equally good.
    """
    print(f"\n--- 1. Confusion Matrix Analysis: {method_name} ---")
    
    cm = confusion_matrix(df["ground_truth"], df["prediction"], labels=["A", "B", "tie", "neither"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["A", "B", "tie", "neither"])
    
    fig, ax = plt.subplots(figsize=(6, 4))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title(f"Confusion Matrix: {method_name}")
    plt.show()
    
    acc = (df["ground_truth"] == df["prediction"]).mean() * 100
    print(f"Overall Accuracy: {acc:.2f}%")

In [ ]:
def analyze_turn_complexity(df):
    """
    ANALYSIS 2: Complexity Scaling (Turn Count)
    ---------------------------------------------
    What it does: Calculates the model's accuracy grouped by the number of conversation turns.
    
    How to interpret for Report: 
    - If accuracy drops significantly as turn count increases (e.g., 80% on 1-turn, 
      but only 40% on 4-turns), it reveals a "context window" or "attention" weakness. 
    - It implies the model struggles to track long conversations and likely only evaluates 
      the final message while forgetting the user's original instructions.
    """
    print("\n--- 2. Complexity Scaling (Accuracy by Turn Count) ---")
    
    if 'num_turns' not in df.columns:
        print("Skipping: 'num_turns' column not found. Ensure you saved it during validation.")
        return
        
    turn_accuracy = df.groupby('num_turns')['correct'].mean() * 100
    turn_counts = df.groupby('num_turns').size()
    
    for turns, acc in turn_accuracy.items():
        count = turn_counts[turns]
        print(f"Conversations with {turns} Turns: {acc:.1f}% accuracy (Sample size: {count})")

In [ ]:
def analyze_internal_contradictions(df):
    """
    ANALYSIS 3: Internal Contradictions (CoT Prompts)
    ---------------------------------------------
    What it does: Scans the raw text response to see if the model's logic contradicts 
    its own final parsed prediction.
    
    How to interpret for Report: 
    - High contradiction counts reveal a weakness in "instruction following" or "recency bias".
    - It means the model might reason correctly in its critique, but hallucinate the wrong 
      letter at the end, or your parsing logic failed to capture the model's true intent.
    """
    print("\n--- 3. Internal Contradictions (Reasoning vs. Verdict) ---")
    
    contradictions = []
    
    for idx, row in df.iterrows():
        pred = str(row['prediction']).upper()
        resp = str(row['full_response']).lower()
        
        is_contradiction = False
        # Checking if it picked A, but explicitly said "output b" or "verdict: b" in text
        if pred == "A" and ("verdict: b" in resp or "output b" in resp):
            is_contradiction = True
        elif pred == "B" and ("verdict: a" in resp or "output a" in resp):
            is_contradiction = True
        elif pred == "TIE" and ("verdict: a" in resp or "verdict: b" in resp):
            is_contradiction = True
            
        if is_contradiction:
            contradictions.append(row)
            
    contradiction_rate = (len(contradictions) / len(df)) * 100
    print(f"Suspected Contradictions found: {len(contradictions)} ({contradiction_rate:.1f}% of data)")
    
    if len(contradictions) > 0:
        print("\nExample Contradiction:")
        example = contradictions[0]
        print(f"  Ground Truth: [{example['ground_truth']}]")
        print(f"  Extracted Prediction: [{example['prediction']}]")
        print(f"  Model's actual ending text: ...{str(example['full_response'])[-120:].replace(chr(10), ' ')}")

In [ ]:
def run_all_analyses(csv_path, method_name):
	print(f"\n\n========== RUNNING COMPLETE ERROR ANALYSIS FOR: {method_name} ==========")
	try:
		df = pd.read_csv(csv_path)
	except FileNotFoundError:
		print(f"File not found: {csv_path}. Please run validation first!")
		return
		
	# Standard Analysis
	analyze_confusion_matrix(df, method_name)
	analyze_turn_complexity(df)
	
	# Check for contradictions in reasoning_1
	if 'reasoning_1' in df.columns:
		print("\n--- 3. Internal Contradictions (Reasoning vs. Verdict) ---")
		df['full_response'] = df['reasoning_1'] # Alias for compatibility
		analyze_internal_contradictions(df)

# --- RUN ANALYSIS 1: GEMMA ---
CSV_1 = "output/validation_gemma2-9b-it-cot_baseline.csv"
run_all_analyses(CSV_1, "Gemma-2-9b (Chain-of-Thought)")

# --- RUN ANALYSIS 2: GLM ---
CSV_2 = "output/validation_glm4.1v-9b-thinking-zero_shot_baseline.csv"
run_all_analyses(CSV_2, "GLM-4.1v-9b-Thinking (Zero-Shot)")
